# Day 16 — Extract reusable functions into Python modules
This notebook installs shared modules and creates **new refactored copies** of your Day 13–15 notebooks.
It does not fit new models, tune models, or run the 2025 test. Original notebooks/results remain in place.

Save this notebook in `finance-volatility/notebooks/`, confirm the path below, and run in order.
Use the same environment as Day 13–15 (pandas, NumPy, scikit-learn, matplotlib, joblib).

**What a module is:** a `.py` file containing functions. A notebook imports them instead of pasting
the same definitions repeatedly. Folder names organize files; `__init__.py` marks the package.

In [1]:
from pathlib import Path
import ast
import hashlib
import importlib
import json
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/Users/yaasarpong/Desktop/PHD/Year 4 fall/finance-volatility')
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
SRC_DIR = PROJECT_ROOT / 'src' / 'finance_project'
assert (PROJECT_ROOT / 'data' / 'financial_raw_data.csv').is_file(), 'Confirm PROJECT_ROOT and your saved data file.'
NAMES = ['day13_machine_learning.ipynb', 'day14_final_test.ipynb', 'day15_robustness_checks.ipynb']
for name in NAMES:
    assert (NOTEBOOKS_DIR / name).is_file(), f'Move your existing notebook into notebooks/: {name}'
original_paths = [NOTEBOOKS_DIR / name for name in NAMES]
original_paths += [p for p in (PROJECT_ROOT / 'results').glob('day1[345]_*') if p.is_file()]
original_hashes = {str(p): hashlib.sha256(p.read_bytes()).hexdigest() for p in original_paths}
original_notebooks = {name: json.loads((NOTEBOOKS_DIR / name).read_text()) for name in NAMES}
print('Project:', PROJECT_ROOT)
print('Original notebooks and saved outputs fingerprinted.')

Project: /Users/yaasarpong/Desktop/PHD/Year 4 fall/finance-volatility
Original notebooks and saved outputs fingerprinted.


## 1. Shared modules
The next cell defines the files to install. You can inspect the source after it runs.

| File in `src/finance_project/` | Responsibility |
|---|---|
| `features.py` | Original predictors and aligned future response construction |
| `models.py` | Training-only scaling and fitting caller-selected rows |
| `evaluation.py` | MAE, RMSE, bias, negative forecasts, benchmark improvements |
| `plots.py` | Figure saving with an explicit output folder |

Data splitting, tuning, frozen choices, and experiment orchestration stay in notebooks.

In [2]:
MODULES = {'__init__.py': '"""Shared calculations extracted from the Day 13–15 finance notebooks."""\n', 'features.py': '"""Feature and response construction; no model fitting or file writes."""\nimport numpy as np\nimport pandas as pd\n\ndef build_features(raw_frame, until="2024-12-31"):\n    """Build original Day 13–15 features through `until` on observed S&P dates.\n\n    Input is a DataFrame with datetime `date`, `sp500`, `vix`, `treasury_10yr`.\n    Returns a new DataFrame. No missing VIX/yield values are filled. Returns\n    use percentage points; rolling windows include the forecast origin.\n    Future shifts construct response labels only, never predictor values.\n    """\n    data = raw_frame.copy()\n    assert {\'date\', \'sp500\', \'vix\', \'treasury_10yr\'}.issubset(data.columns)\n    assert data[\'date\'].notna().all(), \'Missing/invalid dates.\'\n    assert not data[\'date\'].duplicated().any(), \'Duplicate dates.\'\n    data = data.loc[data[\'date\'] <= pd.Timestamp(until)].sort_values(\'date\').reset_index(drop=True)\n    base_columns = [\'sp500\', \'vix\', \'treasury_10yr\']\n    for col in base_columns:\n        data[col] = pd.to_numeric(data[col], errors=\'raise\')\n    assert not np.isinf(data[base_columns].to_numpy()).any()\n    data = data.dropna(subset=[\'sp500\']).reset_index(drop=True)\n    assert data[\'sp500\'].gt(0).all()\n\n    data[\'log_return\'] = 100 * np.log(data[\'sp500\']).diff()\n    data[\'absolute_return\'] = data[\'log_return\'].abs()\n    data[\'squared_return\'] = data[\'log_return\'] ** 2\n    data[\'return_lag_1\'] = data[\'log_return\'].shift(1)\n    data[\'abs_lag_1\'] = data[\'absolute_return\'].shift(1)\n    data[\'abs_lag_5\'] = data[\'absolute_return\'].shift(5)\n    for window in [5, 20]:\n        data[f\'mean_abs_{window}\'] = data[\'absolute_return\'].rolling(window, min_periods=window).mean()\n        data[f\'return_std_{window}\'] = data[\'log_return\'].rolling(window, min_periods=window).std(ddof=1)\n    data[\'treasury_change\'] = data[\'treasury_10yr\'].diff()\n    data[\'target_next_day\'] = data[\'absolute_return\'].shift(-1)\n    data[\'target_date\'] = data[\'date\'].shift(-1)\n    data[\'historical_mean\'] = data[\'absolute_return\'].expanding(min_periods=20).mean()\n    data[\'ewm_20\'] = data[\'absolute_return\'].ewm(span=20, adjust=False, min_periods=20).mean()\n\n    return data\n\ndef make_task(features, horizon, target_kind, ml_features, baseline_columns):\n    """Align point/mean future targets and retain the original common sample.\n\n    Feature eligibility stays fixed across ablation scenarios, as in Day 15.\n    `horizon` counts observed S&P rows. Fit only labels matured by the origin.\n    """\n    assert isinstance(horizon, int) and horizon >= 1\n    frame = features.copy()\n    frame[\'outcome_date\'] = frame[\'date\'].shift(-horizon)\n    if target_kind == \'point\':\n        frame[\'actual\'] = frame[\'absolute_return\'].shift(-horizon)\n        frame[\'outcome_start_date\'] = frame[\'outcome_date\']\n    elif target_kind == \'mean\':\n        future_returns = pd.concat(\n            [frame[\'absolute_return\'].shift(-j) for j in range(1, horizon + 1)], axis=1)\n        frame[\'actual\'] = future_returns.mean(axis=1, skipna=False)\n        frame[\'outcome_start_date\'] = frame[\'date\'].shift(-1)\n    else:\n        raise ValueError(target_kind)\n    frame = frame.dropna(subset=ml_features + baseline_columns +\n                        [\'actual\', \'outcome_date\', \'outcome_start_date\']).copy()\n    assert (frame[\'date\'] < frame[\'outcome_start_date\']).all()\n    assert (frame[\'outcome_start_date\'] <= frame[\'outcome_date\']).all()\n    assert np.isfinite(frame[ml_features + baseline_columns + [\'actual\']].to_numpy()).all()\n    return frame\n', 'models.py': '"""Training-only scaling and unchanged estimator fitting."""\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\n\ndef scaled_model(estimator):\n    """Wrap an estimator so scaling is fitted inside each training fold."""\n    return Pipeline([(\'scaler\', StandardScaler()), (\'model\', estimator)])\n\ndef fit_model(model, frame, feature_names, target_name):\n    """Fit the supplied estimator on caller-selected, already audited rows.\n\n    The notebook owns the temporal split and frozen parameters. This function\n    does not choose rows, tune hyperparameters, or inspect evaluation outcomes.\n    It returns the same fitted estimator.\n    """\n    return model.fit(frame[feature_names], frame[target_name])\n', 'evaluation.py': '"""Paired forecast scores, preserving negative predictions and bias sign."""\nimport numpy as np\nimport pandas as pd\n\ndef score_predictions(frame, methods, target=\'actual\', references=(\'ewm_20\',), reference_labels=None):\n    """Return one row per method: MAE/RMSE/Bias and benchmark improvements.\n\n    Every method uses the supplied shared rows. Bias is forecast minus actual.\n    Predictions are scored unchanged. Returns columns, not a method index.\n    The caller controls ordering/indexing to match its original notebook.\n    """\n    assert len(frame) > 0, \'No aligned forecasts to score.\'\n    assert np.isfinite(frame[[target, *methods]].to_numpy()).all()\n    rows = []\n    for method in methods:\n        error = frame[method] - frame[target]\n        rows.append({\'method\': method, \'n_forecasts\': len(frame),\n                     \'MAE\': error.abs().mean(),\n                     \'RMSE\': np.sqrt((error ** 2).mean()),\n                     \'Bias\': error.mean(),\n                     \'negative_forecasts\': int((frame[method] < 0).sum())})\n    scores = pd.DataFrame(rows)\n    reference_labels = reference_labels or {}\n    for reference in references:\n        for metric in [\'MAE\', \'RMSE\']:\n            denominator = scores.set_index(\'method\').loc[reference, metric]\n            assert denominator > 0\n            label = reference_labels.get(reference, reference)\n            scores[f\'{metric}_improvement_vs_{label}_pct\'] = 100 * (\n                1 - scores[metric] / denominator)\n    return scores\n', 'plots.py': '"""Figure output with an explicit destination directory."""\nfrom pathlib import Path\nimport matplotlib.pyplot as plt\n\ndef save_figure(fig, filename, figures_dir, show=True):\n    """Save a 300-dpi figure, optionally display it, then close it."""\n    destination = Path(figures_dir) / filename\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    fig.tight_layout()\n    fig.savefig(destination, dpi=300, bbox_inches=\'tight\')\n    if show:\n        plt.show()\n    plt.close(fig)\n    return destination\n'}
EXPECTED = {'day13_machine_learning.ipynb': {'save_figure': "def save_figure(fig, filename):\n    fig.tight_layout()\n    fig.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches='tight')\n    plt.show()\n    plt.close(fig)", 'scaled_model': "def scaled_model(estimator):\n    return Pipeline([('scaler', StandardScaler()), ('model', estimator)])"}, 'day14_final_test.ipynb': {'save_figure': "def save_figure(fig, filename):\n    fig.tight_layout()\n    fig.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches='tight')\n    plt.show()\n    plt.close(fig)", 'scaled_model': "def scaled_model(estimator):\n    return Pipeline([('scaler', StandardScaler()), ('model', estimator)])", 'build_data': "def build_data(until):\n    data = pd.read_csv(file_path, parse_dates=['date'])\n    assert {'date', 'sp500', 'vix', 'treasury_10yr'}.issubset(data.columns)\n    assert data['date'].notna().all(), 'Missing/invalid dates.'\n    assert not data['date'].duplicated().any(), 'Duplicate dates.'\n    data = data.loc[data['date'] <= pd.Timestamp(until)].sort_values('date').reset_index(drop=True)\n    base_columns = ['sp500', 'vix', 'treasury_10yr']\n    for col in base_columns:\n        data[col] = pd.to_numeric(data[col], errors='raise')\n    assert not np.isinf(data[base_columns].to_numpy()).any()\n    data = data.dropna(subset=['sp500']).reset_index(drop=True)\n    assert data['sp500'].gt(0).all()\n\n    data['log_return'] = 100 * np.log(data['sp500']).diff()\n    data['absolute_return'] = data['log_return'].abs()\n    data['squared_return'] = data['log_return'] ** 2\n    data['return_lag_1'] = data['log_return'].shift(1)\n    data['abs_lag_1'] = data['absolute_return'].shift(1)\n    data['abs_lag_5'] = data['absolute_return'].shift(5)\n    for window in [5, 20]:\n        data[f'mean_abs_{window}'] = data['absolute_return'].rolling(window, min_periods=window).mean()\n        data[f'return_std_{window}'] = data['log_return'].rolling(window, min_periods=window).std(ddof=1)\n    data['treasury_change'] = data['treasury_10yr'].diff()\n    data['target_next_day'] = data['absolute_return'].shift(-1)\n    data['target_date'] = data['date'].shift(-1)\n    data['historical_mean'] = data['absolute_return'].expanding(min_periods=20).mean()\n    data['ewm_20'] = data['absolute_return'].ewm(span=20, adjust=False, min_periods=20).mean()\n\n    return data", 'score_forecasts': "def score_forecasts(frame):\n    rows = []\n    for name in methods:\n        error = frame[name] - frame['target_next_day']\n        rows.append({'method': name, 'n_forecasts': len(frame),\n                     'MAE': error.abs().mean(), 'RMSE': np.sqrt((error ** 2).mean()),\n                     'Bias': error.mean(), 'negative_forecasts': int((frame[name] < 0).sum())})\n    scores = pd.DataFrame(rows).set_index('method')\n    for reference in ['ewm_20', 'day12_extended']:\n        for metric in ['MAE', 'RMSE']:\n            denominator = scores.loc[reference, metric]\n            assert denominator > 0\n            scores[f'{metric}_improvement_vs_{reference}_pct'] = 100 * (1 - scores[metric] / denominator)\n    return scores.sort_values('RMSE')"}, 'day15_robustness_checks.ipynb': {'save_figure': "def save_figure(fig, filename):\n    fig.tight_layout()\n    fig.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches='tight')\n    plt.show()\n    plt.close(fig)", 'scaled_model': "def scaled_model(estimator):\n    return Pipeline([('scaler', StandardScaler()), ('model', estimator)])", 'build_features': 'def build_features(raw_frame, until="2024-12-31"):\n    data = raw_frame.copy()\n    assert {\'date\', \'sp500\', \'vix\', \'treasury_10yr\'}.issubset(data.columns)\n    assert data[\'date\'].notna().all(), \'Missing/invalid dates.\'\n    assert not data[\'date\'].duplicated().any(), \'Duplicate dates.\'\n    data = data.loc[data[\'date\'] <= pd.Timestamp(until)].sort_values(\'date\').reset_index(drop=True)\n    base_columns = [\'sp500\', \'vix\', \'treasury_10yr\']\n    for col in base_columns:\n        data[col] = pd.to_numeric(data[col], errors=\'raise\')\n    assert not np.isinf(data[base_columns].to_numpy()).any()\n    data = data.dropna(subset=[\'sp500\']).reset_index(drop=True)\n    assert data[\'sp500\'].gt(0).all()\n\n    data[\'log_return\'] = 100 * np.log(data[\'sp500\']).diff()\n    data[\'absolute_return\'] = data[\'log_return\'].abs()\n    data[\'squared_return\'] = data[\'log_return\'] ** 2\n    data[\'return_lag_1\'] = data[\'log_return\'].shift(1)\n    data[\'abs_lag_1\'] = data[\'absolute_return\'].shift(1)\n    data[\'abs_lag_5\'] = data[\'absolute_return\'].shift(5)\n    for window in [5, 20]:\n        data[f\'mean_abs_{window}\'] = data[\'absolute_return\'].rolling(window, min_periods=window).mean()\n        data[f\'return_std_{window}\'] = data[\'log_return\'].rolling(window, min_periods=window).std(ddof=1)\n    data[\'treasury_change\'] = data[\'treasury_10yr\'].diff()\n    data[\'target_next_day\'] = data[\'absolute_return\'].shift(-1)\n    data[\'target_date\'] = data[\'date\'].shift(-1)\n    data[\'historical_mean\'] = data[\'absolute_return\'].expanding(min_periods=20).mean()\n    data[\'ewm_20\'] = data[\'absolute_return\'].ewm(span=20, adjust=False, min_periods=20).mean()\n\n    return data', 'make_task': "def make_task(features, horizon, target_kind):\n    frame = features.copy()\n    frame['outcome_date'] = frame['date'].shift(-horizon)\n    if target_kind == 'point':\n        frame['actual'] = frame['absolute_return'].shift(-horizon)\n        frame['outcome_start_date'] = frame['outcome_date']\n    elif target_kind == 'mean':\n        future_returns = pd.concat(\n            [frame['absolute_return'].shift(-j) for j in range(1, horizon + 1)], axis=1)\n        frame['actual'] = future_returns.mean(axis=1, skipna=False)\n        frame['outcome_start_date'] = frame['date'].shift(-1)\n    else:\n        raise ValueError(target_kind)\n    frame = frame.dropna(subset=ml_features + baseline_columns +\n                        ['actual', 'outcome_date', 'outcome_start_date']).copy()\n    assert (frame['date'] < frame['outcome_start_date']).all()\n    assert (frame['outcome_start_date'] <= frame['outcome_date']).all()\n    assert np.isfinite(frame[ml_features + baseline_columns + ['actual']].to_numpy()).all()\n    return frame", 'evaluate': "def evaluate(frame):\n    rows = []\n    for method in methods:\n        error = frame[method] - frame['actual']\n        rows.append({'method': method, 'n_forecasts': len(frame),\n                     'MAE': error.abs().mean(), 'RMSE': np.sqrt((error ** 2).mean()),\n                     'Bias': error.mean(), 'negative_forecasts': int((frame[method] < 0).sum())})\n    scores = pd.DataFrame(rows)\n    for metric in ['MAE', 'RMSE']:\n        reference = scores.set_index('method').loc['ewm_20', metric]\n        assert reference > 0\n        scores[f'{metric}_improvement_vs_ewm_pct'] = 100 * (1 - scores[metric] / reference)\n    return scores"}}
ORIGINAL_DAY13_FEATURES = "data = pd.read_csv(file_path, parse_dates=['date'])\nassert {'date', 'sp500', 'vix', 'treasury_10yr'}.issubset(data.columns)\nassert data['date'].notna().all(), 'Missing/invalid dates.'\nassert not data['date'].duplicated().any(), 'Duplicate dates.'\ndata = data.loc[data['date'] <= pd.Timestamp('2024-12-31')].sort_values('date').reset_index(drop=True)\nbase_columns = ['sp500', 'vix', 'treasury_10yr']\nfor col in base_columns:\n    data[col] = pd.to_numeric(data[col], errors='raise')\nassert not np.isinf(data[base_columns].to_numpy()).any()\ndata = data.dropna(subset=['sp500']).reset_index(drop=True)\nassert data['sp500'].gt(0).all()\n\ndata['log_return'] = 100 * np.log(data['sp500']).diff()\ndata['absolute_return'] = data['log_return'].abs()\ndata['squared_return'] = data['log_return'] ** 2\ndata['return_lag_1'] = data['log_return'].shift(1)\ndata['abs_lag_1'] = data['absolute_return'].shift(1)\ndata['abs_lag_5'] = data['absolute_return'].shift(5)\nfor window in [5, 20]:\n    data[f'mean_abs_{window}'] = data['absolute_return'].rolling(window, min_periods=window).mean()\n    data[f'return_std_{window}'] = data['log_return'].rolling(window, min_periods=window).std(ddof=1)\ndata['treasury_change'] = data['treasury_10yr'].diff()\ndata['target_next_day'] = data['absolute_return'].shift(-1)\ndata['target_date'] = data['date'].shift(-1)\ndata['historical_mean'] = data['absolute_return'].expanding(min_periods=20).mean()\ndata['ewm_20'] = data['absolute_return'].ewm(span=20, adjust=False, min_periods=20).mean()\n\n"
ORIGINAL_DAY13_SCORES = "score_rows = []\nfor name in methods:\n    error = predictions[name] - predictions['target_next_day']\n    score_rows.append({'method': name, 'n_forecasts': len(error),\n                       'MAE': error.abs().mean(), 'RMSE': np.sqrt((error ** 2).mean()),\n                       'Bias': error.mean(), 'negative_forecasts': int((predictions[name] < 0).sum())})\nperformance = pd.DataFrame(score_rows).set_index('method').sort_values('RMSE')\nfor reference in ['ewm_20', 'day12_extended']:\n    for metric in ['MAE', 'RMSE']:\n        performance[f'{metric}_improvement_vs_{reference}_pct'] = 100 * (\n            1 - performance[metric] / performance.loc[reference, metric])\n"
for filename in MODULES:
    print(filename)

__init__.py
features.py
models.py
evaluation.py
plots.py


## 2. Prepare and inspect refactored copies
This transformation checks that the extracted functions match the versions supplied earlier.
If you changed their calculations, it stops before writing files. Markdown notes and interpretation
cells are retained; copied output cells are cleared so old results are not displayed as a new run.

In [3]:
def canonical(source):
    return ast.dump(ast.parse(source), include_attributes=False)

class Refactor(ast.NodeTransformer):
    def __init__(self, notebook):
        self.notebook = notebook
        self.changed = []

    def visit_FunctionDef(self, node):
        if node.name in EXPECTED[self.notebook]:
            actual = ast.dump(ast.Module(body=[node], type_ignores=[]), include_attributes=False)
            assert actual == canonical(EXPECTED[self.notebook][node.name]), (
                f'{node.name} in {self.notebook} differs from the supplied version. '
                'No files have been changed; this needs a manual review.')
            self.changed.append(node.name)
            return None
        return self.generic_visit(node)

    def visit_Call(self, node):
        node = self.generic_visit(node)
        if isinstance(node.func, ast.Name):
            name = node.func.id
            if name == 'save_figure':
                node.args.append(ast.Name(id='NEW_FIGURES_DIR', ctx=ast.Load()))
            elif name == 'build_data':
                until = node.args[0]
                node.func.id = 'build_features'
                node.args = [ast.Call(func=ast.Attribute(value=ast.Name(id='pd', ctx=ast.Load()),
                             attr='read_csv', ctx=ast.Load()), args=[ast.Name(id='file_path', ctx=ast.Load())],
                             keywords=[ast.keyword(arg='parse_dates', value=ast.List(
                                 elts=[ast.Constant(value='date')], ctx=ast.Load()))]), until]
            elif name == 'make_task':
                node.args.extend([ast.Name(id='ml_features', ctx=ast.Load()),
                                  ast.Name(id='baseline_columns', ctx=ast.Load())])
            elif name in ['evaluate', 'score_forecasts']:
                node.func.id = 'score_predictions'
                node.args.append(ast.Name(id='methods', ctx=ast.Load()))
                if name == 'evaluate':
                    node.keywords.append(ast.keyword(arg='reference_labels', value=ast.Dict(
                        keys=[ast.Constant(value='ewm_20')], values=[ast.Constant(value='ewm')])))
                if name == 'score_forecasts':
                    node.keywords.extend([
                        ast.keyword(arg='target', value=ast.Constant(value='target_next_day')),
                        ast.keyword(arg='references', value=ast.Tuple(elts=[ast.Constant(value='ewm_20'),
                                       ast.Constant(value='day12_extended')], ctx=ast.Load()))])
                    node = ast.Call(func=ast.Attribute(value=node, attr='set_index', ctx=ast.Load()),
                                    args=[ast.Constant(value='method')], keywords=[])
                    node = ast.Call(func=ast.Attribute(value=node, attr='sort_values', ctx=ast.Load()),
                                    args=[ast.Constant(value='RMSE')], keywords=[])
        # Redirect file writes only; all source reads and fingerprints retain original paths.
        if isinstance(node.func, ast.Attribute) and node.func.attr in ['to_csv', 'write_text']:
            if node.func.attr == 'write_text':
                node.func.value = WritePath().visit(node.func.value)
            elif node.args:
                node.args[0] = WritePath().visit(node.args[0])
        if isinstance(node.func, ast.Attribute) and isinstance(node.func.value, ast.Name):
            if node.func.value.id == 'joblib' and node.func.attr == 'dump' and len(node.args) > 1:
                node.args[1] = WritePath().visit(node.args[1])
            # Day 14/15 estimator fitting now uses an explicit shared fitting function.
            elif node.func.value.id == 'model' and node.func.attr == 'fit' and len(node.args) == 2:
                x, y = node.args
                if isinstance(x, ast.Subscript) and isinstance(y, ast.Subscript):
                    if ast.dump(x.value) == ast.dump(y.value):
                        node = ast.Call(func=ast.Name(id='fit_model', ctx=ast.Load()),
                                        args=[node.func.value, x.value, x.slice, y.slice], keywords=[])
        return node

class WritePath(ast.NodeTransformer):
    def visit_Name(self, node):
        if node.id == 'RESULTS_DIR':
            node.id = 'NEW_RESULTS_DIR'
        elif node.id in ['frozen_path', 'protocol_path']:
            return ast.BinOp(left=ast.Name(id='NEW_RESULTS_DIR', ctx=ast.Load()), op=ast.Div(),
                             right=ast.Attribute(value=node, attr='name', ctx=ast.Load()))
        return node

def prepare_copy(name, original):
    notebook = json.loads(json.dumps(original))
    transformer = Refactor(name)
    path_cell_found = False
    feature_block_found = name != 'day13_machine_learning.ipynb'
    for cell in notebook['cells']:
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source'])
        if source.lstrip().startswith('%pip'):
            cell['outputs'] = []
            cell['execution_count'] = None
            continue
        if name == 'day13_machine_learning.ipynb' and source.startswith('data = pd.read_csv'):
            prefix, suffix = source.split('day12_features = ', 1)
            assert canonical(prefix) == canonical(ORIGINAL_DAY13_FEATURES), 'Day 13 feature block was edited.'
            source = ("data = build_features(pd.read_csv(file_path, parse_dates=['date']), '2024-12-31')\n"
                      "base_columns = ['sp500', 'vix', 'treasury_10yr']\n\n"
                      'day12_features = ' + suffix)
            feature_block_found = True
        if name == 'day13_machine_learning.ipynb' and 'score_rows = []' in source:
            start = source.index('score_rows = []')
            end = source.index('display(performance.round(4))', start)
            assert canonical(source[start:end]) == canonical(ORIGINAL_DAY13_SCORES), 'Day 13 scoring block was edited.'
            source = source[:start] + (
                "performance = score_predictions(predictions, methods, target='target_next_day', "
                "references=('ewm_20', 'day12_extended')).set_index('method').sort_values('RMSE')\n"
            ) + source[end:]
        tree = transformer.visit(ast.parse(source))
        ast.fix_missing_locations(tree)
        source = ast.unparse(tree) + '\n'
        if 'PROJECT_ROOT = ' in source:
            assert not path_cell_found
            source += (
                "\nimport sys\n"
                "sys.path.insert(0, str(PROJECT_ROOT / 'src'))\n"
                "from finance_project.features import build_features, make_task\n"
                "from finance_project.models import scaled_model, fit_model\n"
                "from finance_project.evaluation import score_predictions\n"
                "from finance_project.plots import save_figure\n"
                f"NEW_RESULTS_DIR = RESULTS_DIR / 'day16_verification' / '{name[:5]}'\n"
                f"NEW_FIGURES_DIR = FIGURES_DIR / 'day16_verification' / '{name[:5]}'\n"
                "NEW_RESULTS_DIR.mkdir(parents=True, exist_ok=True)\n"
                "NEW_FIGURES_DIR.mkdir(parents=True, exist_ok=True)\n"
            )
            path_cell_found = True
        compile(source, name, 'exec')
        cell['source'] = source.splitlines(keepends=True)
        cell['outputs'] = []
        cell['execution_count'] = None
    assert path_cell_found and feature_block_found
    assert set(transformer.changed) == set(EXPECTED[name])
    intro = ("## Day 16 refactored copy\n"
             "Reusable functions now live in `src/finance_project/`. Original notebook cells and output files "
             "were preserved when creating this copy. New outputs from this copy go to "
             "`results/day16_verification/" + name[:5] + "/` and the corresponding figures subfolder. "
             "Source data, saved configurations, and reproduction checks still use the original files. "
             "Day 13 repeats the original training-only search if fully run; Day 14/15 use saved choices. "
             "Use the Day 16 read-only checks first; full reruns are optional.\n")
    notebook['cells'].insert(0, {'cell_type': 'markdown', 'metadata': {}, 'source': intro.splitlines(keepends=True)})
    return notebook

In [4]:
prepared_copies = {name: prepare_copy(name, original_notebooks[name]) for name in NAMES}
planned = {SRC_DIR / name: source for name, source in MODULES.items()}
for name, notebook in prepared_copies.items():
    destination = NOTEBOOKS_DIR / name.replace('.ipynb', '_refactored.ipynb')
    planned[destination] = json.dumps(notebook, indent=1, ensure_ascii=False) + '\n'
# Preflight ALL destinations before writing any file. A different existing file is preserved.
for destination, content in planned.items():
    if destination.exists():
        assert destination.read_text() == content, f'Existing different file preserved: {destination}'
for destination, content in planned.items():
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists():
        destination.write_text(content, encoding='utf-8')
    print('Ready:', destination.relative_to(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
importlib.invalidate_caches()
from finance_project.features import build_features, make_task
from finance_project.models import scaled_model, fit_model
from finance_project.evaluation import score_predictions
from finance_project.plots import save_figure

Ready: src/finance_project/__init__.py
Ready: src/finance_project/features.py
Ready: src/finance_project/models.py
Ready: src/finance_project/evaluation.py
Ready: src/finance_project/plots.py
Ready: notebooks/day13_machine_learning_refactored.ipynb
Ready: notebooks/day14_final_test_refactored.ipynb
Ready: notebooks/day15_robustness_checks_refactored.ipynb


## 3. Check behavior without rerunning the experiments
These checks use your development data through 2024. They compare the module with the original
feature function and directly verify saved Day 15 prediction scores. They do not use raw 2025 outcomes.

In [5]:
raw = pd.read_csv(PROJECT_ROOT / 'data' / 'financial_raw_data.csv', parse_dates=['date'])
raw = raw.loc[raw['date'] <= pd.Timestamp('2024-12-31')].copy()

def get_original_function(name):
    for cell in original_notebooks['day15_robustness_checks.ipynb']['cells']:
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source'])
        if source.lstrip().startswith('%pip'):
            continue
        for node in ast.parse(source).body:
            if isinstance(node, ast.FunctionDef) and node.name == name:
                return ast.get_source_segment(source, node)
    raise ValueError(name)

env = {'np': np, 'pd': pd}
exec(get_original_function('build_features'), env)
development = build_features(raw)
pd.testing.assert_frame_equal(development, env['build_features'](raw))
config = json.loads((PROJECT_ROOT / 'results' / 'day13_configuration.json').read_text())
ml_features = config['ml_features']
baseline_columns = ['historical_mean', 'ewm_20']
methods = baseline_columns + ['ols', 'lasso', 'random_forest']
env.update(ml_features=ml_features, baseline_columns=baseline_columns, methods=methods)
exec(get_original_function('make_task'), env)
exec(get_original_function('evaluate'), env)
for horizon, kind in [(1, 'point'), (5, 'point'), (5, 'mean')]:
    pd.testing.assert_frame_equal(make_task(development, horizon, kind, ml_features, baseline_columns),
                                  env['make_task'](development, horizon, kind))

input_bytes = development[['date', 'sp500', 'vix', 'treasury_10yr']].to_csv(index=False).encode('utf-8')
assert hashlib.sha256(input_bytes).hexdigest() == config['pre2025_sp500_observation_input_sha256']
predictions = pd.read_csv(PROJECT_ROOT / 'results' / 'day15_validation_predictions.csv')
saved_scores = pd.read_csv(PROJECT_ROOT / 'results' / 'day15_robustness_results.csv')
for scenario, frame in predictions.groupby('scenario', sort=False):
    scores = score_predictions(frame, methods, reference_labels={'ewm_20': 'ewm'})
    pd.testing.assert_frame_equal(scores, env['evaluate'](frame))
    previous = saved_scores.loc[saved_scores['scenario'].eq(scenario)].set_index('method')
    current = scores.set_index('method')
    for column in ['MAE', 'RMSE', 'Bias', 'n_forecasts', 'negative_forecasts',
                   'MAE_improvement_vs_ewm_pct', 'RMSE_improvement_vs_ewm_pct']:
        np.testing.assert_allclose(current[column], previous.loc[current.index, column],
                                   rtol=1e-10, atol=1e-12)

# Future raw observations must not change earlier predictors; labels are intentionally excluded.
boundary = pd.Timestamp('2024-06-30')
modified = raw.copy()
later = modified['date'] > boundary
modified.loc[later, 'sp500'] *= 1.4
modified.loc[later, 'vix'] += 10
modified.loc[later, 'treasury_10yr'] += 1
changed = build_features(modified)
early = development['date'] <= boundary
pd.testing.assert_frame_equal(development.loc[early, ml_features + baseline_columns],
                              changed.loc[early, ml_features + baseline_columns])
for path, fingerprint in original_hashes.items():
    assert hashlib.sha256(Path(path).read_bytes()).hexdigest() == fingerprint
print('PASS: original features, target alignment, development fingerprint, all scenario scores,')
print('future-feature check, and unchanged original notebooks/results.')

PASS: original features, target alignment, development fingerprint, all scenario scores,
future-feature check, and unchanged original notebooks/results.


## 4. Understand the import
Your refactored notebooks now use:

```python
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from finance_project.features import build_features
from finance_project.models import scaled_model, fit_model
from finance_project.evaluation import score_predictions
from finance_project.plots import save_figure
```

Then a former repeated block becomes a call:

```python
data = build_features(raw_data, until="2024-12-31")
scores = score_predictions(predictions, methods, target="actual")
save_figure(fig, "comparison.png", figures_dir=FIGURES_DIR)
```

`target="actual"` is used for Day 15; Day 13/14 use `target="target_next_day"`.
`fit_model(model, train, features, target)` fits the estimator you pass, with your existing rows
and parameters. Training-only time folds and the frozen 2025 procedure remain in the notebooks.

The shared functions need no hidden notebook variables. If a function needs a folder, features,
or methods, those are explicit arguments.

## 5. Finish Day 16
Open the new `_refactored.ipynb` copies and inspect the imports and shorter cells.
Restart a notebook kernel after editing a `.py` module so imports use the current code.
The read-only checks above are the required verification for this extraction step.
Full refactored reruns are optional and save separate outputs under `day16_verification/`.
Keep the original configuration, tuning summary, raw CSV, and completed results together.

This completes the function-extraction portion of Day 16. The broader repository tasks also include
README documentation, dependencies, folder organization, and an appropriate license.